# 05. MCP Server と OAuth ― Amazon Quick から呼べるようにする

5本目では、04 で作った Agent を **外部（Amazon Quick）から呼べるように公開** します。
元教材では Cortex Code スキルの Step 5〜7 にあたる部分です。

| 作るもの | 役割 |
|---|---|
| MCP Server `SALES_OPS_MCP_SERVER` | Agent を MCP（標準プロトコル）のツールとして公開する窓口 |
| OAuth 連携 `SALES_OPS_QUICK_SUITE_OAUTH` | Quick のユーザーが Snowflake にログインするための仕組み |

**前提**：`04_cortex_agent.ipynb` を最後まで実行済みであること

## Step 0. 実行環境をそろえる

使うロール・ウェアハウス・スキーマを指定します。Notebook はそれぞれ独立して動くため、各 Notebook の最初に実行します。

In [ ]:
%%sql -r setup
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE CORTEX_QUICK_HOL_WH;
USE SCHEMA CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA;
SELECT CURRENT_ROLE() AS "ロール", CURRENT_WAREHOUSE() AS "ウェアハウス", CURRENT_SCHEMA() AS "スキーマ";
SELECT CURRENT_USER() AS "ユーザー",
       REPLACE(CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME(), '_', '-') || '.snowflakecomputing.com' AS "アカウントホスト"

> 💡 **アカウントホストについて**：MCP エンドポイントや OAuth の URL は `<組織名>-<アカウント名>.snowflakecomputing.com` で組み立てます。
> アカウント名に `_` が含まれる場合は `-` に置き換えます（URL で使えないため）。

## Step 1. MCP Server を作る

MCP（Model Context Protocol）は、AI エージェントと外部ツールをつなぐ標準プロトコルです。
Snowflake Managed MCP Server を使うと、**サーバーを自前で立てずに** Agent を外部へ公開できます。

- ツールの種類 `CORTEX_AGENT_RUN` で Agent を1つ公開します（外部には Agent だけを見せ、どのツールを使うかは Agent に任せる構成）
- Semantic View を直接公開する `CORTEX_ANALYST_MESSAGE` などの種類もあります

In [ ]:
%%sql -r create_mcp_server
CREATE OR REPLACE MCP SERVER CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_MCP_SERVER
  COMMENT = 'MCP Server exposing Sales Ops Agent for Amazon Quick'
  FROM SPECIFICATION $$
    tools:
      - title: "Sales Ops Agent"
        name: "sales_ops_agent"
        type: "CORTEX_AGENT_RUN"
        identifier: "CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT"
        description: "Ask questions about sales operations data (revenue, pipeline, reps, customers, products, marketing) and the internal sales reports (recommendations, benchmarks)."
  $$

## Step 2. 権限を付ける

MCP Server の USAGE だけでは、配下のツールは使えません。**Agent・Semantic View・Cortex Search の権限も必要**です（02〜04 で付与済み）。
ここでは MCP Server への USAGE を付け、ハンズオン用ロールにそろっているかをまとめて確認します（一覧の `CORTEX_AGENT_SERVER` が MCP Server です）。

> 💡 **元教材との違い**：AWS Workshop 5章 Step 6 では MCP Server への GRANT が2回書かれ、Agent への GRANT が抜けています。本ハンズオンでは 04 で Agent への GRANT を行っています。

In [ ]:
%%sql -r grant_mcp
GRANT USAGE ON MCP SERVER CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_MCP_SERVER TO ROLE CORTEX_QUICK_HOL_ROLE;
-- ハンズオン用ロールに付いている権限を一覧（MCP Server・Agent・Semantic View・Cortex Search の 4 つがあれば OK）
SHOW GRANTS TO ROLE CORTEX_QUICK_HOL_ROLE
  ->> SELECT "privilege", "granted_on", "name" FROM $1
      WHERE "granted_on" IN ('CORTEX_AGENT_SERVER', 'CORTEX_AGENT', 'SEMANTIC_VIEW', 'CORTEX_SEARCH_SERVICE')
      ORDER BY "granted_on", "name"

In [ ]:
%%sql -r show_mcp_servers
SHOW MCP SERVERS IN SCHEMA CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA

## Step 3. 接続ユーザーの既定ロールを設定する

Quick から OAuth でログインしたとき、Snowflake は **そのユーザーのデフォルトロール／デフォルトウェアハウス** で処理します。

- Quick からの処理は、セッションで切り替えたロールではなく **DEFAULT_ROLE** で行われます。
- ACCOUNTADMIN / SECURITYADMIN / ORGADMIN は OAuth でのログインが**ブロック**されるため、DEFAULT_ROLE が ACCOUNTADMIN のままだと Quick から接続できません。
- そこで、Quick でログインするユーザー（＝今このNotebookを実行しているユーザー）の DEFAULT_ROLE を `CORTEX_QUICK_HOL_ROLE` に変えます。

> 💡 変更後も、Snowsight 右上のロール切り替えや `USE ROLE ACCOUNTADMIN` で管理者として作業できます（次回ログイン時の初期ロールが変わるだけです）。
> このNotebookも Step 0 で `USE ROLE ACCOUNTADMIN` を実行しているので影響ありません。元に戻す方法は末尾のクリーンアップを参照してください。

> 💡 あわせて、ユーザーの `ALLOWED_INTERFACES` を `ALL` にします（制限されていると Quick がログイン画面を繰り返し表示します。既定値は `ALL` なので、通常は変化しません）。
>
> 既定ロールを変えたくない場合は、このセルを実行しないでください（その場合 Quick からは接続できません）。

In [ ]:
%%sql -r set_default_role
SET HOL_USER = CURRENT_USER();
ALTER USER IDENTIFIER($HOL_USER) SET DEFAULT_ROLE = CORTEX_QUICK_HOL_ROLE;
ALTER USER IDENTIFIER($HOL_USER) SET DEFAULT_WAREHOUSE = CORTEX_QUICK_HOL_WH;
ALTER USER IDENTIFIER($HOL_USER) SET ALLOWED_INTERFACES = ('ALL');
DESC USER IDENTIFIER($HOL_USER)
  ->> SELECT "property", "value" FROM $1 WHERE "property" IN ('DEFAULT_ROLE', 'DEFAULT_WAREHOUSE', 'ALLOWED_INTERFACES')

## Step 4. OAuth 連携を作る

Amazon Quick のユーザーは OAuth で Snowflake にログインします。ログイン後の処理は、そのユーザーのロールの権限内で行われます（**RBAC がそのまま効く**）。

| 設定 | 値 | 意味 |
|---|---|---|
| `OAUTH_CLIENT_TYPE` | `CONFIDENTIAL` | Client Secret を使う方式 |
| `OAUTH_REDIRECT_URI` | Quick のコールバック URL（us-west-2） | ログイン後の戻り先。**Quick のリージョンと完全一致**が必要 |
| `OAUTH_ALTERNATE_REDIRECT_URIS` | 同（us-east-1） | 追加で受け付ける戻り先。これで us-west-2 / us-east-1 どちらの Quick でも接続できます |
| `OAUTH_ISSUE_REFRESH_TOKENS` | `TRUE` | 再ログインなしで使い続けられるようにする |
| `OAUTH_USE_SECONDARY_ROLES` | `NONE` | DEFAULT_ROLE の権限だけで動かす（MCP 向けの推奨設定）。ユーザーが ACCOUNTADMIN を持っていても、Quick 経由ではハンズオン用ロールの権限に限定されます |
| `BLOCKED_ROLES_LIST` | 既定（ACCOUNTADMIN 等） | 管理者ロールでの OAuth ログインを禁止 |
| `OAUTH_REFRESH_TOKEN_VALIDITY` | `86400` | リフレッシュトークンの有効期間（秒、1日） |

作成には ACCOUNTADMIN が必要です。

> ⚠️ このセルを再実行すると **Client ID / Secret が変わります**。Quick 側に登録した後で再実行した場合は、Quick のコネクタも作り直してください。

In [ ]:
%%sql -r create_oauth
CREATE OR REPLACE SECURITY INTEGRATION SALES_OPS_QUICK_SUITE_OAUTH
  TYPE = OAUTH
  ENABLED = TRUE
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://us-west-2.quicksight.aws.amazon.com/sn/oauthcallback'
  OAUTH_ALTERNATE_REDIRECT_URIS = ('https://us-east-1.quicksight.aws.amazon.com/sn/oauthcallback')
  OAUTH_ISSUE_REFRESH_TOKENS = TRUE
  OAUTH_USE_SECONDARY_ROLES = 'NONE'
  OAUTH_REFRESH_TOKEN_VALIDITY = 86400

## Step 5. Amazon Quick に入力する値を表示する

次のセルで、Quick のコネクタ設定に必要な値をまとめて表示します。手順は `docs/02_quick_setup_guide_ja.md` を参照してください。

> ⚠️ **Client Secret は画面共有で見せないよう注意してください。** 既定では伏せ字で表示し、1 行目を `SET SHOW_SECRET = TRUE;` にしたときだけ全文を表示します。
> Quick に貼り付けるときだけ `TRUE` にして実行し、貼り付け後は `FALSE` に戻して再実行してください。

> 💡 Quick で OAuth ログイン画面が出たら、**この Snowflake アカウントのユーザー名とパスワード**でログインします（MFA を登録済みならその確認も求められます）。

In [ ]:
%%sql -r quick_settings
-- Quick に Client secret を貼り付けるときだけ TRUE にする（貼り付けたら FALSE に戻す）
SET SHOW_SECRET = FALSE;
WITH s AS (SELECT PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('SALES_OPS_QUICK_SUITE_OAUTH')) AS J),
     h AS (SELECT 'https://' || REPLACE(CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME(), '_', '-') || '.snowflakecomputing.com' AS B)
SELECT "項目", "値" FROM (
            SELECT 1 AS O, 'MCP server endpoint' AS "項目",
                   B || '/api/v2/databases/CORTEX_QUICK_HOL_DB/schemas/CORTEX_QUICK_HOL_SCHEMA/mcp-servers/SALES_OPS_MCP_SERVER' AS "値" FROM h
  UNION ALL SELECT 2, 'Client ID', J:OAUTH_CLIENT_ID::STRING FROM s
  UNION ALL SELECT 3, 'Client secret', IFF($SHOW_SECRET, J:OAUTH_CLIENT_SECRET::STRING,
                   LEFT(J:OAUTH_CLIENT_SECRET::STRING, 4) || REPEAT('*', LENGTH(J:OAUTH_CLIENT_SECRET::STRING) - 4)) FROM s
  UNION ALL SELECT 4, 'Token URL', B || '/oauth/token-request' FROM h
  UNION ALL SELECT 5, 'Authorization URL', B || '/oauth/authorize' FROM h
  UNION ALL SELECT 6, 'Redirect URL', 'https://us-west-2.quicksight.aws.amazon.com/sn/oauthcallback または https://us-east-1.quicksight.aws.amazon.com/sn/oauthcallback（Quick 側の表示と一致すれば OK）'
)
ORDER BY O

## Step 6. 設定をまとめて確認する（トラブル時にも使えます）

| 症状 | 原因 | 対処 |
|---|---|---|
| `CREATE AGENT` で `unrecognized field execution_environment` | spec のトップに書いている | `tool_resources.<ツール名>` の中に入れる |
| Agent 実行で `missing an execution environment` | ツールにウェアハウスが紐付いていない | 上と同じ |
| Agent を作り直したら権限が消えた | `CREATE OR REPLACE` は GRANT を引き継がない | GRANT をやり直す |
| Quick から「権限がない」 | DEFAULT_ROLE に権限がない | Step 2・3 を確認 |
| OAuth ログインで拒否される | ACCOUNTADMIN 等は既定でブロック | 既定ロールを `CORTEX_QUICK_HOL_ROLE` に |
| OAuth で `mismatch in the given redirect uri` | Quick のリージョンが登録済み（us-west-2 / us-east-1）以外 | エラー画面の URL の `redirect_uri=` の値を、`ALTER SECURITY INTEGRATION SALES_OPS_QUICK_SUITE_OAUTH SET OAUTH_ALTERNATE_REDIRECT_URIS = ('<その値>')` で追加（Client ID / Secret は変わりません） |
| Quick で 424 / タイムアウト | MCP の呼び出しは 60 秒で打ち切り | 質問を簡単にする、検証済みクエリを増やす |
| Quick がログイン画面を繰り返す（401） | ユーザーの `ALLOWED_INTERFACES` が制限されている | Step 3 のセルで `ALL` に戻す |
| Quick のログインで `invalid_client` | Client ID / Secret の貼り間違い、または OAuth 連携を作り直した | Step 5 の値を貼り直す |
| Quick で Snowflake の URL エラー | アカウント名の `_` をそのまま使っている | Step 5 で表示された URL（`-` に置換済み）を使う |

In [ ]:
%%sql -r check_oauth
DESCRIBE SECURITY INTEGRATION SALES_OPS_QUICK_SUITE_OAUTH
  ->> SELECT "property", "property_value" FROM $1
      WHERE "property" IN ('ENABLED', 'OAUTH_REDIRECT_URI', 'OAUTH_ALTERNATE_REDIRECT_URIS',
                           'OAUTH_CLIENT_TYPE', 'OAUTH_USE_SECONDARY_ROLES', 'BLOCKED_ROLES_LIST')

In [ ]:
%%sql -r check_user
SET HOL_USER = CURRENT_USER();
DESC USER IDENTIFIER($HOL_USER)
  ->> SELECT "property", "value" FROM $1 WHERE "property" IN ('DEFAULT_ROLE', 'DEFAULT_WAREHOUSE', 'ALLOWED_INTERFACES')

Quick と同じロール（`CORTEX_QUICK_HOL_ROLE`）で、MCP Server・Agent・Semantic View を使えるか確かめます。エラーにならず金額が表示されれば OK です。

In [ ]:
%%sql -r check_as_hol_role
USE ROLE CORTEX_QUICK_HOL_ROLE;
DESCRIBE MCP SERVER CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_MCP_SERVER;
DESCRIBE AGENT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT;
SELECT * FROM SEMANTIC_VIEW(CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SEMANTIC_VIEW METRICS TRANSACTIONS.TOTAL_REVENUE)

In [ ]:
%%sql -r back_to_admin
USE ROLE ACCOUNTADMIN

✅ Snowflake 側の準備はここまでです。続きは `docs/02_quick_setup_guide_ja.md` に沿って Amazon Quick 側を設定してください。

---

## クリーンアップ（ハンズオン終了後）

作成したオブジェクトを削除します。誤って実行しないよう、**すべてコメントアウト**してあります。消したい行の先頭の `-- ` を外して実行してください。

In [ ]:
%%sql -r cleanup
-- ① ハンズオンで作ったもの
-- DROP MCP SERVER IF EXISTS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_MCP_SERVER;
-- DROP AGENT IF EXISTS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT;
-- DROP SEMANTIC VIEW IF EXISTS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SEMANTIC_VIEW;
-- DROP SEMANTIC VIEW IF EXISTS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SV_BASIC;
-- DROP CORTEX SEARCH SERVICE IF EXISTS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH;
-- DROP SECURITY INTEGRATION IF EXISTS SALES_OPS_QUICK_SUITE_OAUTH;

-- ② 既定ロールを元に戻す（例：ACCOUNTADMIN）
-- SET HOL_USER = CURRENT_USER();
-- ALTER USER IDENTIFIER($HOL_USER) SET DEFAULT_ROLE = ACCOUNTADMIN;

-- ③ DB・ウェアハウス・ロール・Git 連携まで消す場合（Git から作ったワークスペースは先に削除してください）
-- DROP DATABASE IF EXISTS CORTEX_QUICK_HOL_DB;
-- DROP WAREHOUSE IF EXISTS CORTEX_QUICK_HOL_WH;
-- DROP ROLE IF EXISTS CORTEX_QUICK_HOL_ROLE;
-- DROP DATABASE IF EXISTS QUICK_HOL_GIT_DB;
-- DROP API INTEGRATION IF EXISTS QUICK_HOL_GIT_INTEGRATION;
SELECT 'コメントを外した行だけが実行されます' AS "メモ"